# 07 — Đánh giá CNN đã khóa trên test

In [ ]:
from pathlib import Path
import os, sys
# Bỏ comment nếu Kaggle gắn nhiều bộ code hoặc dữ liệu.
# os.environ['CS231_PROJECT_ROOT'] = '/kaggle/input/TEN-DATASET-CODE'
# os.environ['CS231_DATA_ROOT'] = '/kaggle/input/TEN-DATASET-ANH/DATA'
# os.environ['CS231_CNN_PREVIOUS_RESULTS'] = '/kaggle/input/.../CS231_CNN/results_224'
os.environ.setdefault('MPLCONFIGDIR', '/tmp/cs231-matplotlib')
roots = [Path.cwd(), *Path.cwd().parents]
roots += [p for pattern in ('*', 'datasets/*/*') for p in Path('/kaggle/input').glob(pattern)]
if os.environ.get('CS231_PROJECT_ROOT'):
    roots = [Path(os.environ['CS231_PROJECT_ROOT'])]
projects = sorted({p.resolve() for p in roots if (p/'src/preprocessing.py').is_file()})
if len(projects) != 1:
    raise FileNotFoundError('Đặt CS231_PROJECT_ROOT trỏ tới thư mục chứa src.')
PROJECT_ROOT = projects[0]
sys.path.insert(0, str(PROJECT_ROOT))
from src.experiments.runtime import setup, restore_previous_output
DATA_ROOT, RESULTS_ROOT = setup('CNN', PROJECT_ROOT)

In [ ]:
restore_previous_output(RESULTS_ROOT, 'CNN', 'selection/locked_cnn_config.json')

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from IPython.display import display, Image
from src.CNN.data import CLASSES, CANDIDATES, load_config, configure_gpu, seed_everything
configure_gpu()
seed_everything(42)

## 1. Kiểm tra khóa và mapping

In [ ]:
from src.CNN.pipeline import load_locked_model, evaluate_test
lock = json.loads((RESULTS_ROOT/'selection/locked_cnn_config.json').read_text())
display(pd.DataFrame({'class_index':range(6), 'label_vnd':CLASSES}))
print('Checkpoint:', lock['checkpoint'])

## 2. Chấm toàn bộ test
Argmax softmax, không loại ảnh khó; toàn bộ dự đoán đổi về VND trước khi chấm.

In [ ]:
scores, predictions = evaluate_test(DATA_ROOT, RESULTS_ROOT)
display(pd.DataFrame([{'split':'dev', **lock['dev_metrics']}, {'split':'test', **{k:scores[k] for k in ('accuracy','macro_f1','mae_vnd')}}]))
print('Đúng:', int(predictions.correct.sum()), '/', len(predictions))

## 3. Ma trận nhầm lẫn và từng lớp

In [ ]:
display(Image(filename=str(RESULTS_ROOT/'test_final/confusion_matrix_test.png')))
display(pd.DataFrame(scores['per_class']))
display(pd.DataFrame(scores['error_pairs'], columns=['true_vnd','predicted_vnd','count','gap_vnd','total_gap_vnd']).sort_values('total_gap_vnd', ascending=False))

## 4. Lưu hồ sơ
Dự đoán, metric, môi trường, khóa và hình đã lưu trong test_final. Softmax chưa hiệu chỉnh.